In [ ]:
# ===== Cell 1: Pengaturan =====
import numpy as np, pandas as pd, time
from scipy.optimize import linear_sum_assignment      # hanya VALIDATOR (nilai optimal dihitung Hungarian murni)
from scipy import stats

sizes = [5, 20, 50, 100, 200]
TRIALS = {5: 100, 20: 100, 50: 100, 100: 100, 200: 100}   # jumlah percobaan per ukuran graf
base_seed = 2026                 # seed konfirmasi (123/456/777 dipakai pada tahap eksplorasi)
timing_repeats = 3               # waktu = median dari 3 pengukuran
TOL = 1e-6
ALPHA = 0.05

# Algoritma utama (arahan pembimbing): Greedy Standar, Regret, Regret Terpusat, Hungarian.
include_ngoie = False            # True -> tambahkan hybrid greedy Ngoie dkk. (2022) sebagai pembanding EKSPLORATIF
ablasi_potensial = False         # True -> tambahkan ablasi potensial kolom (rata-rata/minimum/median/maksimum)

# Rencana uji statistik (dua arah, Bonferroni di dalam tiap keluarga; hasil utama = total bobot):
#   F1 (inti, pra-registrasi): Regret Terpusat vs Regret        -> 20 uji (4 kondisi x 5 ukuran)
#   F2 (ditambahkan kemudian): Regret vs Greedy Standar         -> 20 uji
#   F3 (ditambahkan kemudian): Regret Terpusat vs Greedy Standar-> 20 uji
# Catatan: pada pra-registrasi awal F1 berisi 16 uji (n = 20..200); n = 5 ditambahkan sebelum datanya dilihat.
# Aturan keputusan: A "lebih baik" bila p_Bonferroni < 0,05 dan selisih total bobot A-B > 0.

In [ ]:
# ===== Cell 2: Kondisi eksperimen dan seed =====
CONDITIONS = {
    "diskrit_1_100":   (1, lambda r, n: r.integers(1, 101, size=(n, n))),   # bilangan bulat seragam 1..100
    "kontinu_1_100":   (2, lambda r, n: r.uniform(1, 100, size=(n, n))),    # kontinu seragam [1, 100)
    "eksponensial_50": (3, lambda r, n: r.exponential(50, size=(n, n))),    # eksponensial, rata-rata 50
    "banyak_ties_1_5": (4, lambda r, n: r.integers(1, 6, size=(n, n))),     # bilangan bulat 1..5 (banyak bobot sama)
}
run_conditions = list(CONDITIONS)

def make_seed(cond, n, trial):
    """seed unik per (kondisi, ukuran, percobaan)"""
    return base_seed * 10**7 + CONDITIONS[cond][0] * 10**5 + n * 100 + trial

def build_weight_matrix(n, cond, seed):
    return CONDITIONS[cond][1](np.random.default_rng(seed), n)

In [ ]:
# ===== Cell 3: Fungsi dasar =====
def row_top_two(W, i, available_cols):
    """bobot terbesar dan terbesar kedua pada baris i (kolom tersedia)"""
    first, second = -float("inf"), -float("inf")
    for j in available_cols:
        w = W[i][j]
        if w > first:
            second = first
            first = w
        elif w > second:
            second = w
    return first, second

def compute_total_weight(W, matching_pairs):
    """total bobot memakai bobot ASLI W"""
    total = 0
    for i, j in sorted(matching_pairs):
        total += W[i][j]
    return total

In [ ]:
# ===== Cell 4: Algoritma dan pemeriksaan =====
def greedy_standard(W):
    n = W.shape[0]
    selected_left, selected_right, pairs = set(), set(), []
    edge_list = [(-W[i][j], i, j) for i in range(n) for j in range(n)]
    edge_list.sort()                       # bobot menurun; seri: i terkecil, lalu j terkecil
    for neg_w, i, j in edge_list:
        if i not in selected_left and j not in selected_right:
            pairs.append((i, j)); selected_left.add(i); selected_right.add(j)
            if len(pairs) == n: break
    return pairs

def greedy_regret(W):
    """Regret baris: r_i = bobot terbesar - terbesar kedua (kolom tersedia), dihitung ulang tiap iterasi.
    Seri -> baris dengan indeks terkecil; kolom seri -> indeks terkecil."""
    n = W.shape[0]
    selected_left, selected_right, pairs = set(), set(), []
    while len(pairs) < n:
        available_cols = [j for j in range(n) if j not in selected_right]
        best_score, chosen_node = -float("inf"), None
        for i in range(n):
            if i in selected_left: continue
            first, second = row_top_two(W, i, available_cols)
            score = first if len(available_cols) == 1 else first - second
            if score > best_score:
                best_score, chosen_node = score, i
        best_col = max(available_cols, key=lambda j: W[chosen_node][j])
        pairs.append((chosen_node, best_col))
        selected_left.add(chosen_node); selected_right.add(best_col)
    return pairs

def regret_terpusat(W):
    """REGRET TERPUSAT. Pada setiap iterasi, dengan baris tersedia R (m = |R|) dan kolom tersedia C:
         v_j  = (1/m) * sum_{i in R} W[i][j]      (rata-rata kolom)
         W'_ij = W_ij - v_j                        (bobot terpusat)
         r_i  = W'_(1) - W'_(2) pada baris i       (regret pada bobot terpusat)
       Pilih baris i* dengan r terbesar dan kolom j* = argmax W'_{i*,j}; pasangkan (i*, j*).
       Total matching dihitung dengan bobot ASLI. Perhitungan memakai skala m*W_ij - jumlah_kolom_j
       (urutan identik dengan W', tanpa pembagian/pembulatan sehingga seri pada data bulat tepat)."""
    n = W.shape[0]
    selected_left, selected_right, pairs = set(), set(), []
    colsum = [sum(W[i][j] for i in range(n)) for j in range(n)]
    while len(pairs) < n:
        rows = [i for i in range(n) if i not in selected_left]
        cols = [j for j in range(n) if j not in selected_right]
        m = len(rows)
        best_score, chosen_node, chosen_col = -float("inf"), None, None
        for i in rows:
            first, second, col1 = -float("inf"), -float("inf"), None
            for j in cols:
                y = m * W[i][j] - colsum[j]
                if y > first:
                    second = first; first = y; col1 = j
                elif y > second:
                    second = y
            score = 0 if m == 1 else first - second
            if score > best_score:
                best_score, chosen_node, chosen_col = score, i, col1
        pairs.append((chosen_node, chosen_col))
        selected_left.add(chosen_node); selected_right.add(chosen_col)
        for j in cols: colsum[j] -= W[chosen_node][j]
    return pairs

def hungarian_pure(W):
    """Hungarian murni (bentuk potensial, O(n^3)); biaya = max(W) - W"""
    n = W.shape[0]
    cost = W.max() - W
    INF = float("inf")
    u = [0.0] * (n + 1); v = [0.0] * (n + 1)
    p = [0] * (n + 1); way = [0] * (n + 1)
    for i in range(1, n + 1):
        p[0] = i; j0 = 0
        minv = [INF] * (n + 1); used = [False] * (n + 1)
        while True:
            used[j0] = True; i0 = p[j0]; delta = INF; j1 = 0
            for j in range(1, n + 1):
                if not used[j]:
                    cur = float(cost[i0 - 1][j - 1]) - u[i0] - v[j]
                    if cur < minv[j]: minv[j] = cur; way[j] = j0
                    if minv[j] < delta: delta = minv[j]; j1 = j
            for j in range(n + 1):
                if used[j]: u[p[j]] += delta; v[j] -= delta
                else: minv[j] -= delta
            j0 = j1
            if p[j0] == 0: break
        while True:
            j1 = way[j0]; p[j0] = p[j1]; j0 = j1
            if j0 == 0: break
    return sorted((p[j] - 1, j - 1) for j in range(1, n + 1))

def scipy_validator(W):
    r, c = linear_sum_assignment(W.max() - W)
    return list(zip(r.tolist(), c.tolist()))

def ngoie_hybrid(W):
    """EKSPLORATIF: hybrid greedy Ngoie dkk. (2022), tafsiran versi penugasan. Biaya c = max(W)-W. Tiap iterasi:
       reduksi baris lalu kolom (langkah 0 Hungarian); regret garis = terkecil kedua - terkecil; pilih garis dengan regret
       terbesar (seri: baris dahulu, indeks terkecil); pasangkan sel bernilai nol pertama pada garis itu."""
    n = W.shape[0]; mx = W.max()
    C = [[float(mx - W[i][j]) for j in range(n)] for i in range(n)]
    rows, cols, pairs = list(range(n)), list(range(n)), []
    while len(rows) > 1:
        S = [[C[i][j] for j in cols] for i in rows]
        for r in S:
            mn = min(r)
            for k in range(len(r)): r[k] -= mn
        for k in range(len(cols)):
            mn = min(S[a][k] for a in range(len(rows)))
            for a in range(len(rows)): S[a][k] -= mn
        def regret(line):
            a, b = float("inf"), float("inf")
            for x in line:
                if x < a: b = a; a = x
                elif x < b: b = x
            return b - a
        rr = [regret(S[a]) for a in range(len(rows))]
        cr = [regret([S[a][k] for a in range(len(rows))]) for k in range(len(cols))]
        ir = max(range(len(rows)), key=lambda a: (rr[a], -a)); jc = max(range(len(cols)), key=lambda k: (cr[k], -k))
        if rr[ir] >= cr[jc]:
            a = ir; k = next(k for k in range(len(cols)) if S[a][k] <= 1e-12)
        else:
            k = jc; a = next(a for a in range(len(rows)) if S[a][k] <= 1e-12)
        pairs.append((rows[a], cols[k])); rows.pop(a); cols.pop(k)
    pairs.append((rows[0], cols[0]))
    return pairs

# ---- varian numpy: hanya untuk validasi implementasi dan (opsional) ablasi potensial ----
POTENSIAL = {"rata-rata": None,
             "minimum":  lambda S: S.min(axis=0),
             "median":   lambda S: np.median(S, axis=0),
             "maksimum": lambda S: S.max(axis=0)}      # analog reduksi Hungarian pada bobot (biaya = max - bobot)
def regret_pot_np(W, pot):
    W = W.astype(float); n = W.shape[0]; ra = np.arange(n); ca = np.arange(n); tot = 0.0
    while len(ra):
        S0 = W[np.ix_(ra, ca)]
        S = len(ra) * S0 - S0.sum(axis=0)[None, :] if pot == "rata-rata" else S0 - POTENSIAL[pot](S0)[None, :]
        if len(ra) == 1: i = 0
        else:
            srt = -np.sort(-S, axis=1); i = int(np.argmax(srt[:, 0] - srt[:, 1]))
        j = int(np.argmax(S[i])); tot += W[ra[i], ca[j]]; ra = np.delete(ra, i); ca = np.delete(ca, j)
    return tot

# ---- pemeriksaan ----
_W = np.array([[67, 93, 99, 18, 84], [58, 87, 98, 97, 48], [74, 33, 47, 97, 26],
               [84, 79, 37, 97, 81], [69, 50, 56, 68, 3]])
assert compute_total_weight(_W, greedy_standard(_W)) == 356
assert compute_total_weight(_W, hungarian_pure(_W)) == 438
print("Contoh K5,5 | Greedy Standar =", compute_total_weight(_W, greedy_standard(_W)),
      "| Regret =", compute_total_weight(_W, greedy_regret(_W)),
      "| Regret Terpusat =", compute_total_weight(_W, regret_terpusat(_W)),
      "| Optimal =", compute_total_weight(_W, hungarian_pure(_W)))

rng = np.random.default_rng(1); ok_np = ok_geser = 0
for k in range(40):
    W = rng.integers(1, 9, size=(9, 9)) if k % 2 else rng.uniform(1, 50, size=(9, 9))
    ok_np += abs(compute_total_weight(W, regret_terpusat(W)) - regret_pot_np(W, "rata-rata")) < TOL
    Wi = rng.integers(1, 100, size=(10, 10)); a = rng.integers(-50, 50, size=10); b = rng.integers(-50, 50, size=10)
    ok_geser += sorted(regret_terpusat(Wi)) == sorted(regret_terpusat(Wi + a[:, None] + b[None, :]))
print(f"Uji implementasi Regret Terpusat: Python murni vs numpy {ok_np}/40 | invarian terhadap geser a_i+b_j {ok_geser}/40")
assert ok_np == 40 and ok_geser == 40
for f in (greedy_standard, greedy_regret, regret_terpusat, hungarian_pure, scipy_validator, ngoie_hybrid):
    f(build_weight_matrix(5, "diskrit_1_100", 0))       # pemanasan

Contoh K5,5 | Greedy Standar = 356 | Regret = 438 | Regret Terpusat = 438 | Optimal = 438
Uji implementasi Regret Terpusat: Python murni vs numpy 40/40 | invarian terhadap geser a_i+b_j 40/40


In [ ]:
# ===== Cell 5: Eksperimen (perkiraan 45-75 menit) =====
def timed(algo, W, repeats):
    times, pairs = [], None
    for _ in range(repeats):
        t = time.perf_counter()
        pairs = algo(W)
        times.append(time.perf_counter() - t)
    return pairs, float(np.median(times))

results, mismatch_np = [], 0
for cond in run_conditions:
    for n in sizes:
        print(f"\n{'='*100}\nKONDISI: {cond} | UKURAN GRAF: {n} x {n} | PERCOBAAN: {TRIALS[n]}\n{'='*100}", flush=True)
        for trial in range(1, TRIALS[n] + 1):
            seed = make_seed(cond, n, trial)
            W = build_weight_matrix(n, cond, seed)

            ps, t_s = timed(greedy_standard, W, timing_repeats)
            pg, t_g = timed(greedy_regret, W, timing_repeats)
            pt, t_t = timed(regret_terpusat, W, timing_repeats)
            ph, t_h = timed(hungarian_pure, W, timing_repeats)
            pc = scipy_validator(W)
            all_pairs = [ps, pg, pt, ph, pc]
            if include_ngoie:
                pn, t_n = timed(ngoie_hybrid, W, timing_repeats); all_pairs.append(pn)

            for pairs in all_pairs:
                assert sorted(i for i, _ in pairs) == list(range(n))
                assert sorted(j for _, j in pairs) == list(range(n))

            V = {"Greedy": compute_total_weight(W, ps), "Regret": compute_total_weight(W, pg),
                 "RT": compute_total_weight(W, pt), "Optimal": compute_total_weight(W, ph)}
            assert abs(V["Optimal"] - compute_total_weight(W, pc)) < TOL, f"Hungarian murni != SciPy: {cond}, n={n}, trial={trial}"
            for k in ("Greedy", "Regret", "RT"):
                assert V[k] <= V["Optimal"] + TOL

            row = {"kondisi": cond, "n": n, "trial": trial, "seed": seed, **V}
            for k in ("Greedy", "Regret", "RT"):
                row[f"Gap_{k}"] = (V["Optimal"] - V[k]) / V["Optimal"] * 100
            row.update({"Time_Greedy": t_s, "Time_Regret": t_g, "Time_RT": t_t, "Time_Hungarian": t_h})
            if include_ngoie:
                row["Ngoie"] = compute_total_weight(W, pn); row["Gap_Ngoie"] = (V["Optimal"] - row["Ngoie"]) / V["Optimal"] * 100
                row["Time_Ngoie"] = t_n
            if ablasi_potensial:
                for pot in ("rata-rata", "minimum", "median", "maksimum"):
                    row[f"RT_{pot}"] = regret_pot_np(W, pot)
                if abs(row["RT_rata-rata"] - V["RT"]) > TOL: mismatch_np += 1
            results.append(row)
        sub = pd.DataFrame([r for r in results if r["kondisi"] == cond and r["n"] == n])
        print(f"Gap rata-rata %: Greedy {sub.Gap_Greedy.mean():.3f} | Regret {sub.Gap_Regret.mean():.3f} | "
              f"RT {sub.Gap_RT.mean():.3f}   || ms: Regret {sub.Time_Regret.mean()*1e3:.1f}  RT {sub.Time_RT.mean()*1e3:.1f}  "
              f"Hungarian {sub.Time_Hungarian.mean()*1e3:.1f}", flush=True)

df = pd.DataFrame(results)
print("\nSelesai. Baris:", len(df), "| ketidakcocokan RT Python vs numpy (ablasi):", mismatch_np)


KONDISI: diskrit_1_100 | UKURAN GRAF: 5 x 5 | PERCOBAAN: 100
Gap rata-rata %: Greedy 5.060 | Regret 0.708 | RT 0.112   || ms: Regret 0.2  RT 0.2  Hungarian 0.1

KONDISI: diskrit_1_100 | UKURAN GRAF: 20 x 20 | PERCOBAAN: 100
Gap rata-rata %: Greedy 5.213 | Regret 1.548 | RT 0.769   || ms: Regret 2.7  RT 3.4  Hungarian 0.9

KONDISI: diskrit_1_100 | UKURAN GRAF: 50 x 50 | PERCOBAAN: 100
Gap rata-rata %: Greedy 3.511 | Regret 1.004 | RT 0.773   || ms: Regret 44.9  RT 57.6  Hungarian 12.5

KONDISI: diskrit_1_100 | UKURAN GRAF: 100 x 100 | PERCOBAAN: 100
Gap rata-rata %: Greedy 2.381 | Regret 0.722 | RT 0.664   || ms: Regret 149.7  RT 231.6  Hungarian 37.3

KONDISI: diskrit_1_100 | UKURAN GRAF: 200 x 200 | PERCOBAAN: 100
Gap rata-rata %: Greedy 1.524 | Regret 0.429 | RT 0.553   || ms: Regret 1075.6  RT 1627.5  Hungarian 229.1

KONDISI: kontinu_1_100 | UKURAN GRAF: 5 x 5 | PERCOBAAN: 100
Gap rata-rata %: Greedy 5.825 | Regret 1.515 | RT 0.148   || ms: Regret 0.0  RT 0.1  Hungarian 0.0

KONDI

In [ ]:
# ===== Cell 6: Ringkasan, reproduksi, uji statistik, waktu =====
KEYS = ["kondisi", "n"]
cols_gap = ["Gap_Greedy", "Gap_Regret", "Gap_RT"] + (["Gap_Ngoie"] if include_ngoie else [])
nama = {"Gap_Greedy": "Greedy Standar", "Gap_Regret": "Regret", "Gap_RT": "Regret Terpusat", "Gap_Ngoie": "Ngoie (eksploratif)"}

print("=== Gap rata-rata (%) terhadap optimal (Hungarian murni) — makin kecil makin baik ===")
for cond in run_conditions:
    t = df[df.kondisi == cond].groupby("n")[cols_gap].mean().T
    t.index = [nama[c] for c in t.index]; print(f"\n[{cond}]"); print(t.round(3).to_string())

# ---- uji reproduksi terhadap run konfirmasi sebelumnya (seed 2026, n = 20..200; gap Greedy, Regret, RT) ----
REF = {  # kondisi: {n: (Greedy, Regret, RT)}
 "diskrit_1_100":   {20: (5.213, 1.548, 0.769), 50: (3.511, 1.004, 0.773), 100: (2.381, 0.722, 0.664), 200: (1.535, 0.437, 0.550)},
 "kontinu_1_100":   {20: (5.274, 1.408, 0.681), 50: (3.361, 1.125, 0.783), 100: (2.259, 0.666, 0.709), 200: (1.489, 0.364, 0.572)},
 "eksponensial_50": {20: (3.064, 1.195, 0.615), 50: (3.077, 1.486, 0.831), 100: (2.811, 1.298, 0.960), 200: (2.613, 1.365, 1.005)},
 "banyak_ties_1_5": {20: (5.339, 1.348, 0.161), 50: (2.216, 0.672, 0.008), 100: (1.184, 0.308, 0.006), 200: (0.590, 0.193, 0.003)},
}
if base_seed == 2026 and all(TRIALS[n] == t for n, t in {20: 100, 50: 100, 100: 100, 200: 30}.items()):
    m = df.groupby(KEYS)[["Gap_Greedy", "Gap_Regret", "Gap_RT"]].mean()
    bad = [(c, n, k, round(m.loc[(c, n), k], 3), r) for c, d in REF.items() for n, ref in d.items()
           for k, r in zip(["Gap_Greedy", "Gap_Regret", "Gap_RT"], ref) if (c, n) in m.index and abs(m.loc[(c, n), k] - r) > 0.0006]
    print("\nREPRODUKSI terhadap run sebelumnya (n = 20..200):", "SEMUA COCOK" if not bad else f"BERBEDA di {bad}")

def uji(d):
    """Uji berpasangan dua arah pada d = V_A - V_B. Shapiro-Wilk p>=0,05 -> t berpasangan; selain itu Wilcoxon."""
    d = np.asarray(d, float); n = len(d); mean = d.mean(); sd = d.std(ddof=1)
    if np.allclose(d, 0): return dict(mean=mean, lo=mean, hi=mean, uji="-", p=1.0, dz=0.0)
    p_sw = stats.shapiro(d).pvalue if n >= 3 and sd > 0 else 0.0
    if p_sw >= ALPHA: name, p = "t", stats.ttest_1samp(d, 0.0).pvalue
    else:             name, p = "Wilcoxon", stats.wilcoxon(d, zero_method="wilcox").pvalue
    h = stats.t.ppf(0.975, n - 1) * sd / np.sqrt(n)
    return dict(mean=mean, lo=mean - h, hi=mean + h, uji=name, p=p, dz=mean / sd if sd > 0 else 0.0)

def keluarga(nama_k, A, B, kA, kB):
    rows = []
    for (cond, n), g in df.groupby(KEYS):
        r = uji(g[kA].values - g[kB].values)
        r.update(keluarga=nama_k, A=A, B=B, kondisi=cond, n=n, N=len(g),
                 menang=int((g[kA] > g[kB] + TOL).sum()), seri=int((abs(g[kA] - g[kB]) <= TOL).sum()),
                 kalah=int((g[kA] < g[kB] - TOL).sum()))
        rows.append(r)
    out = pd.DataFrame(rows); k = len(out); out["p_bonf"] = (out.p * k).clip(upper=1.0)
    out["hasil"] = np.where((out.p_bonf < ALPHA) & (out["mean"] > 0), f"{A} lebih baik",
                   np.where((out.p_bonf < ALPHA) & (out["mean"] < 0), f"{B} lebih baik", "tidak berbeda nyata"))
    return out

F1 = keluarga("F1", "Regret Terpusat", "Regret", "RT", "Regret")
F2 = keluarga("F2", "Regret", "Greedy Standar", "Regret", "Greedy")
F3 = keluarga("F3", "Regret Terpusat", "Greedy Standar", "RT", "Greedy")
fams = [(F1, "F1 (INTI): Regret Terpusat vs Regret"), (F2, "F2 (tambahan): Regret vs Greedy Standar"),
        (F3, "F3 (tambahan): Regret Terpusat vs Greedy Standar")]
if include_ngoie:
    FN = keluarga("FN", "Ngoie", "Regret Terpusat", "Ngoie", "RT"); fams.append((FN, "FN (EKSPLORATIF): Ngoie vs Regret Terpusat"))
if ablasi_potensial:
    F4 = pd.concat([keluarga("F4", "RT rata-rata", f"RT {p}", "RT_rata-rata", f"RT_{p}") for p in ("minimum", "median", "maksimum")], ignore_index=True)
    F4["p_bonf"] = (F4.p * len(F4)).clip(upper=1.0)
    F4["hasil"] = np.where((F4.p_bonf < ALPHA) & (F4["mean"] > 0), "RT rata-rata lebih baik",
                  np.where((F4.p_bonf < ALPHA) & (F4["mean"] < 0), "pembanding lebih baik", "tidak berbeda nyata"))
    fams.append((F4, "F4 (ablasi potensial, 60 uji)"))

kol = ["kondisi", "n", "N", "mean", "lo", "hi", "uji", "p", "p_bonf", "dz", "menang", "seri", "kalah", "hasil"]
fmt = {"mean": "{:.2f}".format, "lo": "{:.2f}".format, "hi": "{:.2f}".format, "p": "{:.4f}".format, "p_bonf": "{:.4f}".format, "dz": "{:.2f}".format}
for F, judul in fams:
    print(f"\n=== {judul} — selisih total bobot A-B (positif = A lebih baik); Bonferroni: {len(F)} uji, alpha = {ALPHA} ===")
    print(F[kol].to_string(index=False, formatters=fmt))
    print("Ringkasan:", F.hasil.value_counts().to_dict())

print("\n=== Waktu komputasi rata-rata (ms); median dari", timing_repeats, "pengukuran per percobaan ===")
tcols = ["Time_Greedy", "Time_Regret", "Time_RT", "Time_Hungarian"] + (["Time_Ngoie"] if include_ngoie else [])
tm = df.groupby(KEYS)[tcols].mean() * 1e3
tm["RT/Regret"] = tm.Time_RT / tm.Time_Regret
print(tm.round(2).to_string())

=== Gap rata-rata (%) terhadap optimal (Hungarian murni) — makin kecil makin baik ===

[diskrit_1_100]
n                  5      20     50     100    200
Greedy Standar   5.060  5.213  3.511  2.381  1.524
Regret           0.708  1.548  1.004  0.722  0.429
Regret Terpusat  0.112  0.769  0.773  0.664  0.553

[kontinu_1_100]
n                  5      20     50     100    200
Greedy Standar   5.825  5.274  3.361  2.259  1.447
Regret           1.515  1.408  1.125  0.666  0.405
Regret Terpusat  0.148  0.681  0.783  0.709  0.560

[eksponensial_50]
n                  5      20     50     100    200
Greedy Standar   2.296  3.064  3.077  2.811  2.564
Regret           0.578  1.195  1.486  1.298  1.382
Regret Terpusat  0.043  0.615  0.831  0.960  1.020

[banyak_ties_1_5]
n                  5      20     50     100    200
Greedy Standar   5.005  5.339  2.216  1.184  0.568
Regret           1.285  1.348  0.672  0.308  0.164
Regret Terpusat  0.045  0.161  0.008  0.006  0.002

=== F1 (INTI): Regret Ter

In [ ]:
# ===== Cell 7: Ekspor =====
ringkas = df.groupby(KEYS)[cols_gap + ["Time_Greedy", "Time_Regret", "Time_RT", "Time_Hungarian"]].agg(["mean", "std"])
with pd.ExcelWriter("hasil_konfirmasi_final.xlsx") as xw:
    df.to_excel(xw, sheet_name="mentah", index=False)
    ringkas.to_excel(xw, sheet_name="ringkasan")
    pd.concat([F for F, _ in fams]).to_excel(xw, sheet_name="uji_statistik", index=False)
    tm.to_excel(xw, sheet_name="waktu_ms")
df.to_csv("hasil_konfirmasi_final_mentah.csv", index=False)
print("Disimpan: hasil_konfirmasi_final.xlsx dan hasil_konfirmasi_final_mentah.csv")
try:   # Colab: unduh ke komputer (file di runtime hilang jika sesi di-reset)
    from google.colab import files
    files.download("hasil_konfirmasi_final.xlsx"); files.download("hasil_konfirmasi_final_mentah.csv")
except Exception as e:
    print("Unduh otomatis tidak tersedia:", type(e).__name__)

Disimpan: hasil_konfirmasi_final.xlsx dan hasil_konfirmasi_final_mentah.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ===== Cell 8: Spesifikasi sistem (jalankan di sesi YANG SAMA dengan eksperimen) =====
import platform, sys, os, numpy, scipy, pandas
print(platform.platform())
print(sys.version)
print("NumPy", numpy.__version__, "| SciPy", scipy.__version__, "| pandas", pandas.__version__)
print("Jumlah inti:", os.cpu_count())
with open("/proc/cpuinfo") as f:
    print([l.split(":")[1].strip() for l in f if "model name" in l][0])
with open("/proc/meminfo") as f:
    print(f.readline().strip())

Linux-6.6.122+-x86_64-with-glibc2.39
3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
NumPy 2.1.3 | SciPy 1.16.3 | pandas 2.2.3
Jumlah inti: 2
Intel(R) Xeon(R) CPU @ 2.20GHz
MemTotal:       13286944 kB


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving hasil konfirmasi final mentah.csv to hasil konfirmasi final mentah.csv
Saving hasil konfirmasi final trial 100.xlsx to hasil konfirmasi final trial 100.xlsx


In [ ]:
import pandas as pd

df = pd.read_excel(
    'hasil konfirmasi final trial 100.xlsx',
    sheet_name='mentah'
)

assert len(df) == 2000, f"Jumlah baris tidak sesuai: {len(df)}"

df.to_csv('hasil_konfirmasi_final_mentah.csv', index=False)

print("CSV berhasil dibuat.")
print("Ukuran data:", df.shape)

CSV berhasil dibuat.
Ukuran data: (2000, 15)


In [ ]:
# ===== Cell 9: Muat ulang data final (jika df hilang) + uji kecocokan =====
UJI_CEPAT = False      # True = uji asap (n = 5, 20 dan <= 3 percobaan); False = run penuh
df = pd.read_csv("hasil_konfirmasi_final_mentah.csv")
assert len(df) == 2000, len(df)
KEYS = ["kondisi", "n"]
dfx = df[(df.n.isin([5, 20])) & (df.trial <= 3)].copy() if UJI_CEPAT else df.copy()

cek = pd.concat([df[df.n.isin([5, 20])].sample(60, random_state=0),
                 df[df.n == 100].groupby("kondisi").head(1)])
for _, r in cek.iterrows():
    W = build_weight_matrix(int(r.n), r.kondisi, int(r.seed))
    for nama, f in (("Greedy", greedy_standard), ("Regret", greedy_regret), ("RT", regret_terpusat)):
        assert abs(compute_total_weight(W, f(W)) - r[nama]) < TOL, (nama, r.kondisi, r.n, r.trial)
    ro, co = linear_sum_assignment(W.max() - W)
    assert abs(W[ro, co].sum() - r.Optimal) < TOL
print("Cell 9: matriks dibangkitkan ulang dari seed dan cocok dengan CSV (64 instans).")

Cell 9: matriks dibangkitkan ulang dari seed dan cocok dengan CSV (64 instans).


In [ ]:
# ===== Cell 10: Baris vs kolom (RT dan Regret pada W dan W^T) =====
rc = []
for (cond, n), g in dfx.groupby(KEYS, sort=False):
    for _, r in g.iterrows():
        W = build_weight_matrix(int(n), cond, int(r.seed))
        Wt = np.ascontiguousarray(W.T)       # total pada Wt sama dengan total pada bobot asli W
        rc.append(dict(kondisi=cond, n=int(n), trial=int(r.trial), Optimal=r.Optimal,
                       RT=r.RT, RT_T=compute_total_weight(Wt, regret_terpusat(Wt)),
                       Regret=r.Regret, Regret_T=compute_total_weight(Wt, greedy_regret(Wt))))
    print("selesai:", cond, n, flush=True)
rc = pd.DataFrame(rc)
for nm in ("RT", "Regret"):
    rc["d_" + nm] = rc[nm] - rc[nm + "_T"]                    # orientasi baris - orientasi kolom
    rc["beda_" + nm] = (rc["d_" + nm].abs() > TOL).astype(float)
    rc["absrel_" + nm] = rc["d_" + nm].abs() / rc.Optimal * 100
ring_rc = rc.groupby(KEYS).agg(
    pct_beda_RT=("beda_RT", "mean"), mean_d_RT=("d_RT", "mean"), sd_d_RT=("d_RT", "std"), absrel_RT=("absrel_RT", "mean"),
    pct_beda_Regret=("beda_Regret", "mean"), mean_d_Regret=("d_Regret", "mean"), sd_d_Regret=("d_Regret", "std"),
    absrel_Regret=("absrel_Regret", "mean"))
ring_rc["pct_beda_RT"] *= 100; ring_rc["pct_beda_Regret"] *= 100
ring_rc["F1_selisih_rata2"] = dfx.assign(d=dfx.RT - dfx.Regret).groupby(KEYS).d.mean()   # RT - Regret, orientasi asli
print(ring_rc.round(3).to_string())
rc.to_csv("hasil_baris_vs_kolom_mentah.csv", index=False); ring_rc.to_csv("hasil_baris_vs_kolom_ringkasan.csv")

selesai: diskrit_1_100 5
selesai: diskrit_1_100 20
selesai: diskrit_1_100 50
selesai: diskrit_1_100 100
selesai: diskrit_1_100 200
selesai: kontinu_1_100 5
selesai: kontinu_1_100 20
selesai: kontinu_1_100 50
selesai: kontinu_1_100 100
selesai: kontinu_1_100 200
selesai: eksponensial_50 5
selesai: eksponensial_50 20
selesai: eksponensial_50 50
selesai: eksponensial_50 100
selesai: eksponensial_50 200
selesai: banyak_ties_1_5 5
selesai: banyak_ties_1_5 20
selesai: banyak_ties_1_5 50
selesai: banyak_ties_1_5 100
selesai: banyak_ties_1_5 200
                     pct_beda_RT  mean_d_RT  sd_d_RT  absrel_RT  pct_beda_Regret  mean_d_Regret  sd_d_Regret  absrel_Regret  F1_selisih_rata2
kondisi         n                                                                                                                            
banyak_ties_1_5 5            1.0     -0.010    0.100      0.045             25.0         -0.050        0.833          1.717             0.260
                20          34

In [ ]:
# ===== Cell 11: Ablasi potensial kolom (deskriptif) =====
abl, mismatch = [], 0
for (cond, n), g in dfx.groupby(KEYS, sort=False):
    for _, r in g.iterrows():
        W = build_weight_matrix(int(n), cond, int(r.seed))
        row = dict(kondisi=cond, n=int(n), trial=int(r.trial), Optimal=r.Optimal)
        for pot in POTENSIAL:
            row["RT_" + pot.replace("-", "_")] = regret_pot_np(W, pot)
        if abs(row["RT_rata_rata"] - r.RT) > TOL: mismatch += 1
        abl.append(row)
    print("selesai:", cond, n, flush=True)
abl = pd.DataFrame(abl)
pots = [p.replace("-", "_") for p in POTENSIAL]
for p in pots:
    abl["Gap_" + p] = (abl.Optimal - abl["RT_" + p]) / abl.Optimal * 100
print("ketidakcocokan RT rata-rata (numpy) vs RT (Python murni):", mismatch, "dari", len(abl))
print("\nGap rata-rata (%) per potensial (makin kecil makin baik)")
print(abl.groupby(KEYS)[["Gap_" + p for p in pots]].mean().round(3).to_string())
ws = []
for p in pots[1:]:
    d = abl["RT_" + p] - abl["RT_rata_rata"]
    ws.append(abl.assign(d=d).groupby(KEYS).d.agg(
        **{p + "_lebih_baik": lambda x: int((x > TOL).sum()), p + "_seri": lambda x: int((x.abs() <= TOL).sum()),
           p + "_lebih_buruk": lambda x: int((x < -TOL).sum())}))
print("\nPembanding terhadap rata-rata: jumlah instans potensial lain lebih baik / seri / lebih buruk")
print(pd.concat(ws, axis=1).to_string())
abl.to_csv("hasil_ablasi_mentah.csv", index=False)

selesai: diskrit_1_100 5
selesai: diskrit_1_100 20
selesai: diskrit_1_100 50
selesai: diskrit_1_100 100
selesai: diskrit_1_100 200
selesai: kontinu_1_100 5
selesai: kontinu_1_100 20
selesai: kontinu_1_100 50
selesai: kontinu_1_100 100
selesai: kontinu_1_100 200
selesai: eksponensial_50 5
selesai: eksponensial_50 20
selesai: eksponensial_50 50
selesai: eksponensial_50 100
selesai: eksponensial_50 200
selesai: banyak_ties_1_5 5
selesai: banyak_ties_1_5 20
selesai: banyak_ties_1_5 50
selesai: banyak_ties_1_5 100
selesai: banyak_ties_1_5 200
ketidakcocokan RT rata-rata (numpy) vs RT (Python murni): 0 dari 2000

Gap rata-rata (%) per potensial (makin kecil makin baik)
                     Gap_rata_rata  Gap_minimum  Gap_median  Gap_maksimum
kondisi         n                                                        
banyak_ties_1_5 5            0.045        0.692       0.235         0.564
                20           0.161        1.418       0.896         0.985
                50           0.0

In [ ]:
# ===== Cell 12: Instans terstruktur (eksploratif; kondisi id 5 dan 6) =====
CONDITIONS["terstruktur_aditif"] = (5, lambda r, n: r.uniform(1, 50, size=(n, 1)) + r.uniform(0, 100, size=(1, n)) + r.uniform(0, 10, size=(n, n)))
CONDITIONS["terstruktur_perkalian"] = (6, lambda r, n: r.uniform(1, 10, size=(n, 1)) * r.uniform(1, 10, size=(1, n)) * r.uniform(0.9, 1.1, size=(n, n)))
kond_struktur = ["terstruktur_aditif", "terstruktur_perkalian"]
TR = 3 if UJI_CEPAT else 100
ukuran_struktur = [5, 20] if UJI_CEPAT else sizes
res_s = []
for cond in kond_struktur:
    for n in ukuran_struktur:
        print(f"{cond} n={n} percobaan={TR}", flush=True)
        for trial in range(1, TR + 1):
            seed = make_seed(cond, n, trial)
            W = build_weight_matrix(n, cond, seed)
            ps, pg, pt, ph, pc = greedy_standard(W), greedy_regret(W), regret_terpusat(W), hungarian_pure(W), scipy_validator(W)
            for pairs in (ps, pg, pt, ph, pc):
                assert sorted(i for i, _ in pairs) == list(range(n)) and sorted(j for _, j in pairs) == list(range(n))
            V = {"Greedy": compute_total_weight(W, ps), "Regret": compute_total_weight(W, pg),
                 "RT": compute_total_weight(W, pt), "Optimal": compute_total_weight(W, ph)}
            assert abs(V["Optimal"] - compute_total_weight(W, pc)) < TOL, ("Hungarian murni != SciPy", cond, n, trial)
            for k in ("Greedy", "Regret", "RT"): assert V[k] <= V["Optimal"] + TOL
            row = {"kondisi": cond, "n": n, "trial": trial, "seed": seed, **V}
            for k in ("Greedy", "Regret", "RT"): row["Gap_" + k] = (V["Optimal"] - V[k]) / V["Optimal"] * 100
            res_s.append(row)
df_s = pd.DataFrame(res_s)
print(df_s.groupby(KEYS)[["Gap_Greedy", "Gap_Regret", "Gap_RT"]].mean().round(3).to_string())
print("selisih mutlak rata-rata terhadap optimal")
print(df_s.assign(L_G=df_s.Optimal - df_s.Greedy, L_Reg=df_s.Optimal - df_s.Regret, L_RT=df_s.Optimal - df_s.RT).groupby(KEYS)[["L_G", "L_Reg", "L_RT"]].mean().round(2).to_string())
df_s.to_csv("hasil_terstruktur_mentah.csv", index=False)

terstruktur_aditif n=5 percobaan=100
terstruktur_aditif n=20 percobaan=100
terstruktur_aditif n=50 percobaan=100
terstruktur_aditif n=100 percobaan=100
terstruktur_aditif n=200 percobaan=100
terstruktur_perkalian n=5 percobaan=100
terstruktur_perkalian n=20 percobaan=100
terstruktur_perkalian n=50 percobaan=100
terstruktur_perkalian n=100 percobaan=100
terstruktur_perkalian n=200 percobaan=100
                           Gap_Greedy  Gap_Regret  Gap_RT
kondisi               n                                  
terstruktur_aditif    5         2.364       0.624   0.011
                      20        2.535       1.245   0.078
                      50        1.988       1.135   0.086
                      100       1.514       0.963   0.087
                      200       1.122       0.751   0.066
terstruktur_perkalian 5         0.719       0.708   0.026
                      20        1.501       1.810   0.633
                      50        1.483       2.063   0.798
                      1

In [ ]:
# ===== Cell 13: Pencarian lokal kasus buruk (demonstrasi, BUKAN batas rasio) =====
def optimum(W):
    r, c = linear_sum_assignment(W.max() - W); return W[r, c].sum()
def cari_terburuk(algo, n, restart, iterasi, seed=7, maks=1000):
    rng = np.random.default_rng(seed); terbaik = (np.inf, None)
    for _ in range(restart):
        W = rng.integers(1, maks + 1, size=(n, n)); r = compute_total_weight(W, algo(W)) / optimum(W)
        for _ in range(iterasi):
            V = W.copy()
            for _ in range(int(rng.integers(1, 3))):
                i, j = int(rng.integers(n)), int(rng.integers(n))
                V[i, j] = min(maks, max(1, int(V[i, j]) + int(rng.integers(-200, 201))))
            rr = compute_total_weight(V, algo(V)) / optimum(V)
            if rr <= r: r, W = rr, V
        if r < terbaik[0]: terbaik = (r, W.copy())
    return terbaik
for n in ([3] if UJI_CEPAT else [3, 4, 6, 8]):
    for nama, f in (("RT", regret_terpusat), ("Regret", greedy_regret)):
        r, W = cari_terburuk(f, n, restart=2 if UJI_CEPAT else 6, iterasi=200 if UJI_CEPAT else 3000)
        print(f"n={n} {nama}: rasio terburuk yang ditemukan = {r:.3f}")
        if n == 3: print(W.tolist())

n=3 RT: rasio terburuk yang ditemukan = 0.885
[[733, 111, 1], [925, 88, 620], [1, 212, 732]]
n=3 Regret: rasio terburuk yang ditemukan = 0.633
[[1, 1000, 930], [598, 669, 1], [1, 1000, 929]]
n=4 RT: rasio terburuk yang ditemukan = 0.840
n=4 Regret: rasio terburuk yang ditemukan = 0.589
n=6 RT: rasio terburuk yang ditemukan = 0.774
n=6 Regret: rasio terburuk yang ditemukan = 0.587
n=8 RT: rasio terburuk yang ditemukan = 0.835
n=8 Regret: rasio terburuk yang ditemukan = 0.597


In [ ]:
from google.colab import files
import os

file_hasil = [
    "hasil_baris_vs_kolom_mentah.csv",
    "hasil_baris_vs_kolom_ringkasan.csv",
    "hasil_ablasi_mentah.csv",
    "hasil_terstruktur_mentah.csv",
]

for nama in file_hasil:
    if os.path.exists(nama):
        files.download(nama)
        print(f"Memulai download: {nama}")
    else:
        print(f"File belum ditemukan: {nama}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Memulai download: hasil_baris_vs_kolom_mentah.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Memulai download: hasil_baris_vs_kolom_ringkasan.csv


... (2 outputs left out)

Memulai download: hasil_ablasi_mentah.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Memulai download: hasil_terstruktur_mentah.csv
